In [1]:
# You may need to install these libraries first:
# pip install sentence-transformers scikit-learn numpy

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np



In [ ]:
# Original
# --- 1. Sample Data & Query ---
chunks = [
    "Applicants must be at least 21 years old.", # Relevant
    "Minimum monthly salary is ₹25,000.", # Relevant
    "Applicants must have a credit score above 700.", # Relevant
    "The age requirement is that applicants must be at least 21.", # Redundant
    "A credit score of 700 or higher is mandatory.", # Redundant
    "Personal loans are not available to self-employed applicants.", # Slightly less relevant
    "EMI defaults attract a penalty of 2% per month.", # Irrelevant
]
query = "What are the age, salary, and credit score requirements for a personal loan?"


In [ ]:
# Example-1

# Original
# --- 1. Sample Data & Query ---
chunks = [
    "Bajaj Finance — 38th AGM (2025) ", # Relevant
    "Minimum monthly salary is ₹25,000.", # Relevant
    "Applicants must have a credit score above 700.", # Relevant
    "The age requirement is that applicants must be at least 21.", # Redundant
    "A credit score of 700 or higher is mandatory.", # Redundant
    "Personal loans are not available to self-employed applicants.", # Slightly less relevant
    "EMI defaults attract a penalty of 2% per month.", # Irrelevant
]

#bajaj_finance_agm_2025_summary
chunks = [
    # --- Quick facts / meeting logistics ---
    "Bajaj Finance Limited held its 38th Annual General Meeting (AGM) for FY2024–25.",
    "The meeting took place on Thursday, 24 July 2025, at 3:30 p.m. (IST).",
    "Venue: Hyatt Regency, Viman Nagar / Nagar Road, Pune.",
    "E-voting was available before the meeting; live webcast access was provided.",
    "Formal proceedings included quorum verification, Chairman’s address, presentation of reports, and voting on resolutions.",

    # --- Main resolutions ---
    "Shareholders approved the standalone and consolidated audited financial statements for FY2024–25.",
    "Final dividend for FY25 was approved as proposed by the Board.",
    "Re-appointment of statutory auditors and approval of their remuneration.",
    "Re-appointment of directors whose term was expiring and adoption of reports under the Companies Act.",
    "Shareholders ratified statutory and regulatory disclosures under SEBI and RBI compliance frameworks.",

    # --- Financial results and performance highlights ---
    "FY25 financial performance showed strong topline and profit growth.",
    "Net interest income and AUM grew significantly year-on-year.",
    "Consumer lending, personal loans, and mortgages were major growth drivers.",
    "Asset quality remained stable with controlled GNPA/NNPA ratios.",
    "Profitability was driven by net interest income growth, fee income, and cost management.",
    "The company emphasized diversification across business verticals.",
    "Technology and analytics were highlighted as enablers for scale and risk management.",

    # --- Dividend & capital allocation ---
    "FY25 dividend comprised interim and final components.",
    "Interim/special dividend paid earlier in 2025; final dividend ratified at AGM.",
    "Record and payment dates were shared through investor notices and filings.",
    "Management emphasized balancing shareholder returns and growth capital retention.",
    "Capital adequacy and strong cash flows supported consistent dividend distribution.",

    # --- Management commentary ---
    "Management expressed optimism about FY26 growth while maintaining asset quality.",
    "Strategic focus areas: product distribution expansion, technology enhancement, underwriting discipline.",
    "Emphasis on customer acquisition through digital and physical channels.",
    "Focus on data-driven risk control and automation to enhance efficiency.",
    "Deposits business optimization discussed with rate adjustments and product mix management.",
    "Continued investment in digital transformation and analytics platforms.",

    # --- Shareholder Q&A themes ---
    "Shareholders inquired about dividend amounts, payment timelines, and growth guidance.",
    "Questions were raised on asset quality and provisioning approach.",
    "Management reaffirmed stable GNPA/NNPA ratios and conservative provisioning.",
    "Queries about deposit rate reductions were explained as cost optimization measures.",
    "Regulatory compliance and RBI oversight discussed; management assured strict adherence.",
    "No immediate plans for major acquisitions; focus remains on organic growth.",

    # --- Governance & Board updates ---
    "Re-appointment of independent and non-executive directors approved.",
    "Audit committee and remuneration committee reports were adopted.",
    "Ratification of related-party transactions as per SEBI regulations.",
    "Emphasis on strong corporate governance and transparency.",
    "Annual Report 2024–25 provided committee disclosures and compliance statements.",

    # --- Market and analyst reaction ---
    "Analysts reacted positively to the strong AUM growth and earnings consistency.",
    "Share price showed strength post-Q4 FY25 results and AGM commentary.",
    "Brokerages highlighted Bajaj Finance’s diversified growth and asset quality resilience.",
    "Analysts appreciated clarity on margin management and deposit strategy.",
    "Market sentiment remained bullish post-AGM due to stable outlook.",

    # --- Broader business updates ---
    "AUM growth remained in double digits across consumer, SME, and mortgage segments.",
    "Deposit franchise expanded with focus on retail participation.",
    "Product innovation and partnerships with OEMs continued to drive volume.",
    "Digital channels contributed significantly to new customer acquisition.",
    "Cross-sell and upsell strategies improved customer lifetime value.",
    "Technology-driven credit models enhanced operational scalability.",

    # --- Documents available to shareholders ---
    "Notice of the 38th AGM and Annual Report FY24–25 uploaded on investor relations website.",
    "Q4 FY25 investor presentation and earnings transcript published on April 29, 2025.",
    "Webcast recording of AGM made available for investors.",
    "Stock exchange filings included AGM resolutions and voting results.",
    "Investor FAQs detailed dividend details and procedural information.",

    # --- Key takeaways for shareholders ---
    "Shareholders reaffirmed trust in management through approval of all resolutions.",
    "Dividend payments reinforced the company’s strong cash flow position.",
    "Emphasis on sustainable, controlled growth rather than aggressive expansion.",
    "Continued focus on digital transformation and customer experience.",
    "Company positioned itself as a benchmark NBFC in India for governance and profitability.",

    # --- Risks & investor watchpoints ---
    "Interest rate volatility may affect funding costs and margins.",
    "Macroeconomic slowdown could pressure loan demand and repayment behavior.",
    "Regulatory changes for NBFCs could influence capital adequacy and dividend policy.",
    "Increasing competition from fintechs and banks may compress yields.",
    "Management aims to mitigate risk through technology, analytics, and portfolio diversification.",

    # --- Source references & materials ---
    "Annual Report FY2024–25 and AGM Notice (filed on exchanges).",
    "Investor presentations (Q4 FY25) and conference call transcripts.",
    "Corporate announcements and press releases on dividends and results.",
    "Stock exchange disclosures of voting outcomes and director appointments.",
    "Analyst coverage from major financial media and brokerage houses.",

    # --- Overall conclusion ---
    "The 38th AGM of Bajaj Finance Limited validated the company’s consistent performance and strategic clarity.",
    "Shareholders endorsed management’s focus on sustainable growth, risk control, and digital expansion.",
    "The company continues to maintain industry leadership with robust financial metrics and strong governance.",
    "FY25 demonstrated Bajaj Finance’s resilience, profitability, and customer-centric innovation.",
    "Looking ahead, the company aims for balanced growth with continued shareholder value creation."
]

query = "What are the age, salary, and credit score requirements for a personal loan?"


In [ ]:

# --- 2. Embedding ---
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks)
query_embedding = model.encode([query])

In [ ]:
# --- 3. Vanilla Top-k Retrieval ---
def vanilla_top_k(query_embedding, chunk_embeddings, chunks, k=3):
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_k_idx = similarities.argsort()[-k:][::-1]
    return [chunks[i] for i in top_k_idx], similarities[top_k_idx]

vanilla_results, vanilla_scores = vanilla_top_k(query_embedding, chunk_embeddings, chunks, k=3)
print("--- Vanilla Top-3 Results (Often Redundant) ---")
for text, score in zip(vanilla_results, vanilla_scores):
    print(f"  - (Score: {score:.2f}) {text}")


# --- 4. MMR-based Retrieval ---
def mmr(query_embedding, chunk_embeddings, chunks, k=3, lambda_param=0.5):
    """Returns k chunks using Maximum Marginal Relevance."""
    query_chunk_similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    selected_indices = []
    candidates_indices = list(range(len(chunks)))

    best_idx = np.argmax(query_chunk_similarities)
    selected_indices.append(best_idx)
    candidates_indices.remove(best_idx)

    for _ in range(k - 1):
        if not candidates_indices: break

        mmr_scores = []
        candidate_embeddings = chunk_embeddings[candidates_indices]

        for i, cand_idx in enumerate(candidates_indices):
            relevance_to_query = query_chunk_similarities[cand_idx]

            selected_embeddings = chunk_embeddings[selected_indices]
            max_similarity_to_selected = np.max(cosine_similarity(candidate_embeddings[i:i+1], selected_embeddings))

            score = lambda_param * relevance_to_query - (1 - lambda_param) * max_similarity_to_selected
            mmr_scores.append(score)

        best_candidate_idx = candidates_indices[np.argmax(mmr_scores)]
        selected_indices.append(best_candidate_idx)
        candidates_indices.remove(best_candidate_idx)

    return [chunks[i] for i in selected_indices]

mmr_results = mmr(query_embedding, chunk_embeddings, chunks, k=3, lambda_param=0.7)
print("\n--- MMR Top-3 Results (Relevant and Diverse) ---")
for text in mmr_results:
    print(f"  - {text}")


--- Vanilla Top-3 Results (Often Redundant) ---
  - (Score: 0.57) Applicants must have a credit score above 700.
  - (Score: 0.57) A credit score of 700 or higher is mandatory.
  - (Score: 0.49) Personal loans are not available to self-employed applicants.

--- MMR Top-3 Results (Relevant and Diverse) ---
  - Applicants must have a credit score above 700.
  - Personal loans are not available to self-employed applicants.
  - Minimum monthly salary is ₹25,000.
